# 01 — Exploratory data analysis (train + val only)

**Test isolation:** this notebook never loads test images or test metadata. `eda.load_eda_frame` can only return train/val rows; the only thing shown about test is its per-class count. Run with `make eda` after `make data`.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

from defect_detection.config import load_config
from defect_detection.data import eda

# nbconvert runs from notebooks/; resolve paths from the repo root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
config = load_config(Path(os.environ.get("DD_CONFIG", "configs/train.yaml")))
RAW, PROCESSED = config.data.raw_dir, config.data.processed_dir
FIGURES = Path("reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Split sizes (test: counts only) and exclusions

In [ ]:
eda.split_counts(PROCESSED)

In [ ]:
report = json.loads((PROCESSED / "split_report.json").read_text(encoding="utf-8"))
clusters = report["pool_clustering"]
print("excluded:", report["excluded"])
pd.Series({k: v for k, v in clusters.items() if not isinstance(v, (dict, list))})

In [ ]:
print("cluster sizes:", clusters["cluster_size_histogram"])
print("nearest-neighbour similarity:", clusters["nearest_neighbour_similarity_histogram"])
print("clusters with conflicting labels:", clusters["clusters_with_conflicting_labels"])

Leakage audit (content-based, independent of cluster ids). Only aggregate similarity statistics are shown for the evaluation splits, never their images.

In [ ]:
audit = json.loads((PROCESSED / "leakage_audit.json").read_text(encoding="utf-8"))
print("passed:", audit["passed"], audit["checks"])
pd.DataFrame(audit["cross_split_similarity"]).T

## 2. Class balance (train + val)

In [ ]:
frame = eda.load_eda_frame(PROCESSED)
balance = pd.crosstab(frame["split"], frame["class_name"], margins=True)
balance["defect_rate"] = (balance["defective"] / balance["All"]).round(3)
balance

## 3. Image sizes and modes

In [ ]:
frame.groupby(["width", "height", "mode"]).size().rename("images").to_frame()

In [ ]:
frame["size_bytes"].describe().round(0)

## 4. Lighting-shortcut check

If global brightness or contrast alone separates the classes, the model could learn the capture conditions instead of the defect. `separability_auc` near 0.5 means no shortcut; near 1.0 means a strong one.

In [ ]:
stats = eda.image_stats(frame, RAW)
eda.lighting_shortcut(stats)

In [ ]:
fig = eda.feature_histograms(stats)
fig.savefig(FIGURES / "eda_brightness_contrast.png", dpi=120)

## 5. Sample images per class

In [ ]:
fig = eda.sample_grid(frame, RAW, per_class=8, seed=config.seed)
fig.savefig(FIGURES / "eda_samples.png", dpi=120)

## 6. Largest duplicate clusters (augmented copies / label-noise spot check)

Clusters are groups of exact or rotation/flip-aware near-duplicates. They show whether the raw data already contains augmented variants of the same part.

In [ ]:
fig = eda.cluster_grid(frame, RAW, max_clusters=6)
if fig is not None:
    fig.savefig(FIGURES / "eda_duplicate_clusters.png", dpi=120)

## 7. Findings (train + val only; test seen as counts only)

1. **Clean files.** 6,045 train+val images, all 300×300 RGB JPEG, none corrupt.
2. **Class balance.** 60.4% defective in both train and val (stratification worked); moderate
   imbalance, handled with class-weighted cross-entropy (D-022).
3. **Same-part copies are everywhere.** The 300×300 release is built from augmented copies of a
   smaller set of parts (rotations, flips, small zoom/brightness changes). 2,041 of 6,045
   pool images sit in multi-image clusters (largest: 50). Grouping keeps every cluster in one
   split. The official test shared parts with train (64 byte-identical, 524 more at ≥ 0.99);
   those train images are excluded.
4. **Lighting shortcut (important).** Brightness alone separates the classes with AUC 0.883
   (defective images darker: 139 vs 150); contrast alone gives 0.812. This is a capture
   artefact, not a defect property. The ±20% brightness/contrast jitter exceeds the gap; V4
   checks whether errors track brightness (D-035).
5. **Label noise.** 6 near-duplicate clusters contain both labels: either labelling errors or
   copies just above the 0.99 threshold. They are few (6 of 4,706 clusters) and listed for the
   V4 error analysis.
6. **Residual similarity.** Even after cleaning, the official test's median similarity to
   train is 0.978 vs 0.880 for the 512×512 external set, so the external set is the more
   honest estimate of performance on unseen parts.
